# Longformer — The Long-Document Transformer (2020)

_Papers · Transformers & LLMs_

**Make self-attention scale to thousands of tokens by letting each token look only at a local window plus a few global tokens, turning O(n^2) cost into O(n).**

---

This notebook is a paced, step-by-step walkthrough of the **Longformer — The Long-Document Transformer (2020)** lesson. Run each cell, read the short note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We build Longformer's sliding-window attention from scratch in four steps: (1) the attention primitives and the band mask, (2) a local-signal toy sequence to run them on, (3) checking the window reproduces full attention on in-window neighbours, and (4) the cost, ablation, and global-token demos.

### Step 1 — Build the attention primitives and the band mask

`scores_full` computes the usual scaled dot-product scores `Q·Kᵀ / √d_k` for every token pair. `band_mask` is the heart of Longformer: it allows position pair `(i, j)` only when they are within `w//2` of each other — a sliding window — and additionally lets any *global* token attend to, and be attended by, every position (Section 3.1). `longformer_attention` masks the disallowed scores to `-inf` *before* the softmax so they get exactly zero weight; `full_attention` is the dense O(n²) baseline.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

def scores_full(Q, K):
    d_k = Q.shape[-1]
    scaled = (Q @ K.transpose(-2, -1)) / (d_k ** 0.5)   # (n, n) scaled dot-product scores
    return scaled

def band_mask(n, w, global_idx=()):
    """Allow (i,j) iff |i-j| <= w//2; global tokens attend/are-attended by all. Section 3.1."""
    i = torch.arange(n).unsqueeze(1)
    j = torch.arange(n).unsqueeze(0)
    allowed = (i - j).abs() <= (w // 2)     # sliding-window band: |i-j| within half the window
    for g in global_idx:                    # global tokens get a full row AND a full column
        allowed[g, :] = True
        allowed[:, g] = True
    return allowed

def longformer_attention(Q, K, V, w, global_idx=()):
    S = scores_full(Q, K)
    allowed = band_mask(Q.shape[0], w, global_idx)
    S = S.masked_fill(~allowed, float('-inf'))   # mask BEFORE softmax
    W = S.softmax(dim=-1)                         # dropped pairs get exactly zero weight
    out = W @ V
    return out, W

def full_attention(Q, K, V):
    W = scores_full(Q, K).softmax(dim=-1)
    out = W @ V
    return out, W

### Step 2 — Make a sequence whose signal is mostly local

To give the sliding window a fair test, we build a sequence where each token is correlated mainly with its near neighbours. Starting from random vectors `base`, we smooth them so token `t` is a blend of token `t-1` and a little fresh noise — nearby tokens end up similar, distant tokens largely independent. We use the same tensor for `Q`, `K`, and `V` (self-attention style) and run both the windowed and full attention on it.

In [ ]:
# Toy sequence with MOSTLY-LOCAL signal: token t correlates with its neighbours.
n, d_k, w = 40, 16, 8

base = torch.randn(n, d_k)
local = base.clone()
for t in range(1, n):                       # smooth so nearby tokens are similar
    local[t] = 0.7 * local[t - 1] + 0.3 * base[t]

Q = K = V = local                           # self-attention style: same tensor for Q, K, V

out_w, Wwin = longformer_attention(Q, K, V, w)
out_f, Wfull = full_attention(Q, K, V)

### Step 3 — Check the window reproduces full attention on local signal

If the signal really is local, full attention should already place almost all of its weight inside the window — so the cheap windowed map should closely match the expensive full one. We renormalize the full weights over just the in-window positions and compare them to the windowed weights: the gap should be tiny. We also report how much of full attention's total weight already falls inside the window (close to 1 for a local sequence).

In [ ]:
# Does the WINDOW match FULL attention on the in-window neighbours?
allowed = band_mask(n, w)

# Renormalize full weights over just the in-window positions, then compare to windowed weights.
full_in = Wfull * allowed
full_in = full_in / full_in.sum(-1, keepdim=True)
gap = (full_in - Wwin).abs().max().item()
print("max |full(in-window, renormed) - window| weight gap:", round(gap, 4))   # small

mass_in_window = Wfull.mul(allowed).sum(-1).mean().item()
print("mass full attention already puts inside the window (avg):",
      round(mass_in_window, 4))            # close to 1 for local signal

### Step 4 — Cost, window ablation, and the global token

Three quick demonstrations. First, the cost: full attention scores `n²` pairs while the window scores only ~`n·w`, so the gap widens as `n` grows. Second, an ablation: shrinking `w` collapses how far a *single* layer can reach (its receptive field). Third, the global token: making token 0 global lets it attend to every position and lets every position attend to it — the escape hatch Longformer uses to recover long-range reach.

In [ ]:
# Worked cost numbers: full n^2 vs window ~ n*w.
for nn in (8, 64, 512):
    am = band_mask(nn, 4)
    full_pairs = nn * nn
    window_pairs = int(am.sum())
    print(f"n={nn:>4}: full pairs={full_pairs:>7}, window live pairs={window_pairs:>6}")

# ABLATION: shrink the window -> receptive field of ONE layer collapses.
for ww in (8, 2):
    am = band_mask(n, ww)
    reach = am[n // 2].nonzero().flatten()
    n_reached = int(am[n // 2].sum())
    print(f"w={ww}: middle token reaches {reach.min().item()}..{reach.max().item()} "
          f"({n_reached} tokens in one layer)")

# Global token demo: token 0 made global attends to / is attended by all.
_, Wg = longformer_attention(Q, K, V, w=4, global_idx=(0,))
n_global_attends = int((Wg[0] > 0).sum())
print("global token 0 attends to", n_global_attends, "of", n, "tokens (full row)")
print("everyone attends to token 0:", bool((Wg[:, 0] > 0).all().item()))

## Visualize the data & results

_How does the cost of attention grow with sequence length n — full O(n^2) vs Longformer's sliding-window O(n*w)?_

### Step 1 — Count the live pairs for each sequence length

`band_live_pairs` rebuilds the sliding-window band for a given `n` and `w` and counts how many position pairs survive — these are the only pairs the windowed attention ever scores. We isolate this in its own helper so the comparison loop below stays readable.

In [ ]:
import torch

def band_live_pairs(n, w):
    i = torch.arange(n).unsqueeze(1)
    j = torch.arange(n).unsqueeze(0)
    live = ((i - j).abs() <= (w // 2)).sum()
    return int(live)

### Step 2 — Tabulate full O(n²) vs windowed O(n·w)

We hold the window fixed at `w = 8` and sweep the sequence length. The full column grows quadratically (`n²`) while the windowed column grows only linearly in `n`, so the saving balloons for long documents — the whole point of Longformer.

In [ ]:
w = 8
print("n | full n^2 | window live pairs")
for n in (16, 32, 64, 128, 256, 512):
    full_pairs = n * n
    window_pairs = band_live_pairs(n, w)
    print(f"{n:>4} | {full_pairs:>8} | {window_pairs:>6}")
# full:   256, 1024, 4096, 16384, 65536, 262144   (quadratic)
# window: 124,  268,  556,  1132,  2284,   4588   (linear in n)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** With window $w=6$, how many tokens can a single sliding-window layer attend to, and what is the receptive field after $\ell=4$ stacked layers (no dilation)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- One layer: $\tfrac{1}{2}w=3$ on each side, so $3+3=6$ neighbours plus itself = 7 tokens. — _Window $w$ means $\tfrac{1}{2}w$ per side (Section 3.1)._
- Receptive field after $\ell$ layers $=\ell\times w = 4\times6 = 24$. — _Stacking grows reach linearly with depth._

**Answer:** One layer attends to 7 tokens (6 neighbours + itself); after 4 layers the receptive field spans 24 tokens. Stacking is how a local window reaches far.

</details>

**Problem 2.** For $n=1000$ and window $w=10$, roughly how many scored pairs does full attention need vs sliding-window attention, and what is the ratio?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Full $= n^2 = 1000^2 = 1{,}000{,}000$ pairs. — _Every token attends to all $n$ tokens — $O(n^2)$._
- Window $\approx n\times w = 1000\times10 = 10{,}000$ pairs. — _Each token attends to ~$w$ neighbours — $O(n\times w)$._
- Ratio $= 1{,}000{,}000 / 10{,}000 = 100\times$. — _The saving is $n/w$._

**Answer:** Full needs ~1,000,000 pairs; sliding-window ~10,000 — a 100$\times$ reduction. The saving grows as $n/w$, so it is larger for longer documents.

</details>

**Problem 3.** Ablation: you shrink the window from $w=8$ to $w=2$ in a 3-layer model. What happens to the receptive field, and what kind of dependency can the model no longer capture in 3 layers?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Old receptive field $=\ell\times w = 3\times8 = 24$ tokens; new $=3\times2 = 6$ tokens. — _Receptive field is $\ell\times w$ (Section 3.1)._
- Any dependency between tokens more than 6 apart can no longer be formed by stacking alone. — _Information cannot hop further than the receptive field in $\ell$ layers._
- To recover long range you must either add layers, dilate the window, or add a global token. — _Those are the paper's three reach mechanisms._

**Answer:** Receptive field collapses from 24 to 6 tokens, so dependencies spanning more than 6 positions are lost without extra depth, dilation, or a global token. The CODE ablation prints exactly this collapse — the small window can no longer connect distant tokens.

</details>